# ROA analysis — what moved it, and by how much

Reads the **ROA formula** out of both workbooks, follows it down to the inputs that actually changed,
names them from column B, and writes an interactive page where you tick any combination of components
and the real formula is re-evaluated.

Needs these files in the same folder as this notebook:

| File | Role |
|---|---|
| `formula_trace.py` | Parses the formula, walks its precedents, finds what changed |
| `roa_explorer.py` | Renders the interactive page |
| `vintage_compare.py` | Supplies the row map (the inserted row after 200) |

---

## Before you start: the workbooks must be real `.xlsx`

**Renaming `Report.xlsb` to `Report.xlsx` does not work.** They are different file formats, not
different labels — a renamed `.xlsb` will fail to open or open as nonsense. Convert properly:

> **Excel → File → Save As → Excel Workbook (\*.xlsx)**

This is only needed for the *formula* trace. `pyxlsb` and `xlrd` expose cached values and nothing
else — the formula text simply is not available through them — so a `.xlsb` cannot be traced at all.
The value comparison in `vintage_delta_comparison.ipynb` still works on the original `.xlsb`.

`load_book()` refuses a `.xlsb` with that instruction rather than failing obscurely.

While you are in Excel, **let it recalculate and save**. That stores the cached values, which this
notebook cross-checks its own arithmetic against.

---

## Why ticking components beats a waterfall

ROA is net income ÷ assets — a **ratio** — so its drivers do not add up. On the worked example the
components' individual effects sum to −12.196 bps while the true combined move is −12.055 bps. That
0.141 bps gap is real interaction, and a static waterfall has to either hide it or allocate it
arbitrarily. Re-evaluating the actual formula for whatever subset you select is the only way to answer
*"what did these two changes do together"* correctly.

## What counts as a component

A cell becomes a component when **it changed and nothing labelled below it changed**. So `Net income`
and `Total revenue` are walked *through* rather than reported, and what you get back is the individual
line — `Credit provision`, `Interest income` — under its column B name.

## Two ways the number can move

**Inputs moved.** A revenue or expense line changed. Those are the components — tick them and watch.

**The calculation changed shape.** A formula was rewritten, or — the subtle one — a `SUM` range
*spans the inserted row*, so `SUM(D197:D201)` becomes `SUM(D197:D202)` and silently picks up a cell
that has no counterpart in the old workbook. After the row shift those two formulas read *identically*,
so comparing the text finds nothing; the ranges are compared cell by cell instead.

That second part cannot be attributed to any component, so it is reported separately as
**unattributed bps** rather than being spread across the drivers.

## The integrity check

`Trace.check()` re-evaluates the rebuilt formula with all-old inputs and asserts it reproduces the
workbook's own cached value. If the formula was followed incorrectly it **raises** instead of returning
a plausible wrong number. The *new* side is reported rather than asserted — a gap there is a finding
about the workbooks, not a bug in the trace.

## The month columns

ROA is per month: row 5 holds `M1`…`M24` across columns **D–AA**, and the formula reads the month
number out of it with `MID(D$5,2,4)` to annualise. **Column C is an empty spacer** — there is no
formula in it. The page gives you a sheet selector and a month selector.

## 1. Imports

In [ ]:
import importlib
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import formula_trace as ft
import roa_explorer as rx
import vintage_compare as vc
for m in (ft, rx, vc):
    importlib.reload(m)

try:
    import pandas as pd
except ImportError:
    pd = None

print("loaded from", Path(ft.__file__).parent)
print("formula support:", ", ".join(sorted(set(ft.FUNCTIONS) | ft.LAZY_FUNCTIONS)))

## 2. Settings
`ROA_ROW_OLD` is the row in the **old** workbook. The new one is worked out from the row map, so you
do not enter 210 anywhere — if the map is right, 209 lands on 210 by itself. Section 3 shows you that
it did.

In [ ]:
# ============================================================================
#  EDIT THIS CELL
# ============================================================================

OLD_FILE = "Copy of Acquisitions - Vintage Comparison Report - UAT.xlsx"          # current process
NEW_FILE = "Acquisitions - Vintage Comparison Report - UAT Snowflake DRAFT.xlsx"  # snow process
OUTPUT_PAGE = "roa_explorer.html"

ROA_ROW_OLD = 209        # ROA row in the OLD workbook. The new row comes from the row map.
MONTH_COLS  = "D:AA"     # the 24 month columns. Column C is an empty spacer, no formula in it.
MONTH_ROW   = 5          # where M1..M24 live, which the ROA formula reads with MID(D$5,2,4)
LABEL_COL   = "B"        # component names come from here
MAX_DEPTH   = 25         # how far down the chain to follow. Too shallow now REFUSES rather
                         # than guessing, so raise it if section 4 tells you to.

SHEET_PATTERN = r"vintage\s*0*(\d+)$"   # picks the 24 vintage sheets out of all 46
ONLY_SHEETS   = None                     # or an explicit list to narrow the run
ONLY_MONTHS   = None                     # or e.g. ["M12", "M24"]

# Each vintage carries one month fewer than the last: Vintage1 has 24, Vintage24 has 1.
# Months are detected from the workbook and checked against this, so a mismatch is visible
# rather than silently producing a column of empty results.
def months_on(vintage_number: int) -> int:
    return 25 - vintage_number

# How much of each sheet to read. The chain never looks above row 210 or past column AA,
# and reading only that window is the difference between seconds and minutes on a big file.
MAX_LOAD_ROW = 300
MAX_LOAD_COL = 27        # AA
MAX_CELLS    = 20_000    # ceiling per trace; it stops and says so rather than running away

# ---------------------------------------------------------------------------
import re, time
from openpyxl.utils import column_index_from_string, get_column_letter

_c1, _c2 = MONTH_COLS.split(":")
MONTH_COL_IDS = list(range(column_index_from_string(_c1), column_index_from_string(_c2) + 1))
LABEL_COL_IDX = column_index_from_string(LABEL_COL)

VINTAGE_SPEC = vc.vintage_specs(n_vintages=24)[1]
_ROW_MAP, _ = vc.build_row_map(VINTAGE_SPEC)
def row_map(r: int) -> int:
    """Everything past row 200 sits one lower in the snow workbook."""
    return _ROW_MAP.get(r, r if r <= 200 else r + 1)

print(f"ROA row : old {ROA_ROW_OLD}  ->  new {row_map(ROA_ROW_OLD)}")
print(f"a row before the insertion : old 189 -> new {row_map(189)}  (unshifted, as expected)")
print(f"months  : {len(MONTH_COL_IDS)} columns {_c1}..{_c2}, expecting "
      f"{months_on(1)} on Vintage1 down to {months_on(24)} on Vintage24 "
      f"({sum(months_on(n) for n in range(1, 25))} combinations in total)")
print(f"reading : rows 1-{MAX_LOAD_ROW}, columns A-{get_column_letter(MAX_LOAD_COL)}, "
      f"vintage sheets only")
assert row_map(ROA_ROW_OLD) == ROA_ROW_OLD + 1, "the ROA row should shift by one"
assert row_map(189) == 189, "rows at or below 200 should not shift"

## 3. Check the ROA cell before tracing anything
Prints what is actually in the ROA row of both workbooks: the label in column B, the formula, the
cached value, and which columns along that row even hold formulas.

**What you want to see:** the same label both sides, a formula in `ROA_COL`, and a cached value. If
the cached value is empty, Excel never saved a calculated result — re-save the file. If the formula
sits in a different column, change `ROA_COL` above.

In [ ]:
books, vintage_sheets, months, months_found = {}, [], {}, {}

def vintage_key(name):
    m = re.search(SHEET_PATTERN, name.strip(), re.I)
    return int(m.group(1)) if m else None

missing = [p for p in (OLD_FILE, NEW_FILE) if not Path(p).exists()]
if missing:
    print("NOT FOUND:", *missing, sep="\n   ")
else:
    # Sheet names first, so only the sheets that matter get read.
    names_old, names_new = ft.sheet_names(OLD_FILE), ft.sheet_names(NEW_FILE)
    print(f"old: {len(names_old)} sheets   new: {len(names_new)} sheets")

    wanted_old, wanted_new = [], []
    for name in names_old:
        n = vintage_key(name)
        if n is None or (ONLY_SHEETS is not None and name not in ONLY_SHEETS):
            continue
        other = vc.resolve_sheet(name, names_new)
        if other is None:
            print(f"   no match in the new workbook for {name!r}")
            continue
        vintage_sheets.append((n, name, other))
        wanted_old.append(name); wanted_new.append(other)
    vintage_sheets.sort()
    skipped = len(names_old) - len(wanted_old)
    print(f"{len(vintage_sheets)} vintage sheets matched, {skipped} other sheets not read")

    t0 = time.time()
    for tag, path, want in (("old", OLD_FILE, wanted_old), ("new", NEW_FILE, wanted_new)):
        try:
            books[tag] = ft.load_book(path, sheets=want,
                                      max_row=MAX_LOAD_ROW, max_col=MAX_LOAD_COL)
        except ft.UnsupportedFormula as exc:
            print(f"{tag}: {exc}")
    print(f"read both workbooks in {time.time() - t0:.1f}s")

if len(books) == 2 and vintage_sheets:
    _, s_old, s_new = vintage_sheets[0]
    b = books["old"]
    months = {c: b.value(s_old, MONTH_ROW, c) for c in MONTH_COL_IDS}
    print(f"\nmonth headers on row {MONTH_ROW}: "
          f"{months[MONTH_COL_IDS[0]]} .. {months[MONTH_COL_IDS[-1]]}")
    print(f"\nROA row in {s_old}:")
    print(f"   column {LABEL_COL} says : {b.value(s_old, ROA_ROW_OLD, LABEL_COL_IDX)!r}")

    first = next((c for c in MONTH_COL_IDS if b.formula(s_old, ROA_ROW_OLD, c)), None)
    if first is None:
        print(f"   no formula anywhere on row {ROA_ROW_OLD} - check ROA_ROW_OLD and MONTH_COLS")
    else:
        f_old = b.formula(s_old, ROA_ROW_OLD, first)
        f_new = books["new"].formula(s_new, row_map(ROA_ROW_OLD), first)
        print(f"\n   old {get_column_letter(first)}{ROA_ROW_OLD}:\n      {f_old}")
        print(f"   new {get_column_letter(first)}{row_map(ROA_ROW_OLD)}:\n      {f_new}")
        remapped = ft.remap_rows(f_old, row_map)
        same = re.sub(r"\s+", "", remapped).upper() == re.sub(r"\s+", "", f_new or "").upper()
        print(f"\n   with the row shift applied:\n      {remapped}")
        print(f"   -> {'the same formula, just shifted' if same else 'THE FORMULA ITSELF DIFFERS'}")

    print(f"\nExpecting {months_on(1)} months on Vintage1 down to {months_on(24)} on Vintage24. "
          f"Rather than trusting that, section 4 tries every month column and keeps the ones\n"
          f"whose formula actually produces a value - a month with no data returns \"\" from the\n"
          f"IF(...=\"\",\"\") guard, and is reported as skipped rather than counted.")

## 4. Trace every vintage
Follows the formula on each sheet and checks the result reproduces both cached values. A sheet that
cannot be traced is reported with the reason and skipped, rather than quietly dropping out.

In [ ]:
traces, groups, items, failures = [], [], [], []

if len(books) == 2 and vintage_sheets:
    cols_all = [c for c in MONTH_COL_IDS
                if ONLY_MONTHS is None or str(months.get(c)) in ONLY_MONTHS]
    total = len(vintage_sheets) * len(cols_all)
    print(f"trying {len(cols_all)} month columns on each of {len(vintage_sheets)} sheets; "
          f"months with no data are skipped\n")

    t0, done = time.time(), 0
    for n, s_old, s_new in vintage_sheets:
        cols = cols_all
        got, bad = ft.trace_columns(
            books["old"], books["new"], s_old, ROA_ROW_OLD, cols,
            row_map=row_map, label_col=LABEL_COL_IDX, max_depth=MAX_DEPTH,
            sheet_new=s_new, max_cells=MAX_CELLS)
        for tr in got:
            traces.append(tr); groups.append(s_old); items.append(str(months.get(tr.col, tr.ref_old)))
        failures += [(f"{s_old}!{ref}", why) for ref, why in bad]
        done += len(cols)
        rate = done / max(time.time() - t0, 1e-9)
        want = months_on(n)
        flag = "" if len(got) == want else f"   <-- expected {want}"
        print(f"   {s_old:<14}{len(got):>3} months with data   "
              f"{done}/{total} tried, ~{max(total - done, 0) / max(rate, 1e-9):.0f}s left{flag}",
              flush=True)

    print(f"\n{len(traces)} traced, {len(failures)} skipped, in {time.time() - t0:.1f}s")

    by_sheet = {}
    for tr, g in zip(traces, groups):
        by_sheet.setdefault(g, []).append(tr)
    print(f"\n{'sheet':<14}{'months':>7}{'old ROA':>11}{'new ROA':>11}{'delta':>10}"
          f"{'unattributed':>14}{'comps':>7}   (final month)")
    def pc(x):  return "-" if x is None else f"{x:.4%}"
    def bp(x):  return "-" if x is None else f"{x * 10_000:+.1f}b"
    for sheet, ts in by_sheet.items():
        last = ts[-1]
        print(f"{sheet:<14}{len(ts):>7}{pc(last.value_old):>11}{pc(last.value_new):>11}"
              f"{bp(last.delta):>10}{bp(last.structural_gap):>14}{len(last.components):>7}")

    seen = set()
    for tr in traces:
        for sc in tr.structural:
            k = (sc.label, sc.kind, sc.describe())
            if k in seen:
                continue
            seen.add(k)
            if len(seen) <= 6:
                print(f"\nSTRUCTURAL  {sc.describe()}")
                print(f"   old: {sc.formula_old}")
                print(f"   new: {sc.formula_new}")
    if len(seen) > 6:
        print(f"\n... and {len(seen) - 6} more distinct formula differences (all on the page)")
    if not seen:
        print("\nNo formula differences: every cell in the chain computes the same thing in both "
              "workbooks, so the whole move is attributable to the components.")

    empty = [f for f in failures if "no data for this month" in f[1]]
    real = [f for f in failures if f not in empty]
    if empty:
        print(f"\n{len(empty)} month columns carry no data and were skipped - expected, that is "
              f"the triangle.")
    for ref, why in real[:5]:
        print(f"\nSKIPPED {ref}: {why[:220]}")
else:
    print("Load both workbooks in section 3 first.")

## 5. Write the interactive page
One self-contained HTML file — no assets to send alongside it. Open it, or email it to whoever asked.

In [ ]:
if traces:
    payload = rx.build_payload(traces, names=[f"{g} · {i}" for g, i in zip(groups, items)],
                               groups=groups, items=items, month_of=months)
    page = rx.write_html(
        payload, OUTPUT_PAGE,
        title="What moved ROA, component by component",
        subtitle=f"{Path(OLD_FILE).stem} → {Path(NEW_FILE).stem} · "
                 f"row {ROA_ROW_OLD} / {row_map(ROA_ROW_OLD)}, months "
                 f"{items[0]}–{items[-1]}",
        sample=False)
    mb = page.stat().st_size / 1e6
    print(f"written: {page}  ({mb:.1f} MB)")
    print(f"{len(traces)} sheet-month combinations, selectable on the page.")
    if mb > 8:
        print(f"\nThat is large for an attachment. Narrow ONLY_MONTHS or ONLY_SHEETS in section 2\n"
              f"and re-run if you need to send it rather than open it locally.")
else:
    print("Nothing traced - fix section 4 first.")

## 6. The same thing as a table
Every component across every vintage, and the components that show up again and again — those are the
ones worth explaining to whoever signs this off.

In [ ]:
if not traces:
    print("Run section 4 first.")
elif pd is None:
    print("pandas is not installed - `pip install pandas` to use this section.")
else:
    rows = []
    for tr, g, it in zip(traces, groups, items):
        base = tr.evaluate_with(set())
        for c in tr.components:
            only = tr.evaluate_with({c.key})
            rows.append({"Sheet": g, "Month": it, "Component": c.label,
                         "Cell": c.ref_old, "Old": c.old, "New": c.new, "Change": c.delta,
                         "Solo bps": None if (only is None or base is None)
                                     else (only - base) * 10_000})
    comp = pd.DataFrame(rows)
    print(f"{len(comp):,} changed components over {len(traces):,} sheet-month combinations\n")

    print("Which line items move ROA, across everything")
    display(comp.groupby("Component")
                .agg(sheets=("Sheet", "nunique"), appearances=("Cell", "count"),
                     total_change=("Change", "sum"), worst_bps=("Solo bps", "min"),
                     best_bps=("Solo bps", "max"))
                .sort_values("worst_bps"))

    roa = pd.DataFrame([{
        "Sheet": g, "Month": it,
        "Old ROA": tr.value_old, "New ROA": tr.value_new,
        "Delta bps": None if tr.delta is None else tr.delta * 10_000,
        "From components bps": None if (tr.modelled_new is None or tr.value_old is None)
                               else (tr.modelled_new - tr.value_old) * 10_000,
        "Unattributed bps": None if tr.structural_gap is None else tr.structural_gap * 10_000,
    } for tr, g, it in zip(traces, groups, items)])

    print("\nROA by sheet and month (largest moves first)")
    display(roa.reindex(roa["Delta bps"].abs().sort_values(ascending=False).index).head(25).round(3))

    print("\nHow much of the move the components actually explain")
    display(roa.groupby("Sheet")[["Delta bps", "From components bps", "Unattributed bps"]]
               .sum().round(2).sort_values("Delta bps"))

## 8. Explain one cell
When a figure looks wrong, point this at it. It prints the whole chain underneath: every precedent,
its formula, what each workbook has cached for it, what this code computed, and which cells became
components. That is enough to see by eye where a modelled figure parts company with the sheet.

In [ ]:
EXPLAIN_SHEET = "Vintage3"     # the sheet you want to look at
EXPLAIN_MONTH = "M3"           # the month, as it appears on row 5

if len(books) == 2 and vintage_sheets:
    s_old = vc.resolve_sheet(EXPLAIN_SHEET, books["old"].sheets)
    s_new = vc.resolve_sheet(EXPLAIN_SHEET, books["new"].sheets)
    col = next((c for c, m in months.items() if str(m) == EXPLAIN_MONTH), None)
    if s_old is None or col is None:
        print(f"Could not find {EXPLAIN_SHEET!r} / {EXPLAIN_MONTH!r}. "
              f"Sheets: {[s for _, s, _ in vintage_sheets][:6]}... "
              f"Months: {[str(m) for m in months.values()][:6]}...")
    else:
        ft.explain(books["old"], books["new"], s_old, ROA_ROW_OLD, col,
                   row_map=row_map, label_col=LABEL_COL_IDX,
                   max_depth=MAX_DEPTH, sheet_new=s_new, show_ranges=8)
else:
    print("Load both workbooks in section 3 first.")

## 9. Self-test on generated workbooks
Builds a pair of workbooks carrying your formula chain, with the snow copy one row longer past 200
and an inserted row that lands inside the cost subtotal range. Asserts the trace finds the individual
lines, reproduces the arithmetic, and reports the widened range rather than absorbing it.

In [ ]:
RUN_SELF_TEST = True        # set to False once you are pointing at the real workbooks

if RUN_SELF_TEST:
    import tempfile
    from openpyxl import Workbook
    from openpyxl.utils import get_column_letter as L

    demo_dir = Path(tempfile.mkdtemp(prefix="roa_selftest_"))
    COLS = list(range(4, 28))                       # D..AA = M1..M24

    def build_demo(path, snow):
        """The real chain: 209 -> 207 -> 205 -> 204 -> 202 -> 193 -> 187:192."""
        shift = (lambda r: r + 1 if r > 200 else r) if snow else (lambda r: r)
        R = {n: shift(n) for n in (193, 202, 204, 205, 207, 209)}
        wb = Workbook(); wb.remove(wb.active)
        for v in (1, 2):
            ws = wb.create_sheet(f"Vintage{v}")
            def put(r, c, val=None, f=None, label=None):
                if label is not None:
                    ws.cell(row=r, column=2, value=label)
                ws.cell(row=r, column=c, value=f if f else val)
            for i, c in enumerate(COLS, start=1):
                X = L(c)
                put(5, c, f"M{i}", label="Month")
                put(27, c, 1.0, label="Active flag")
                put(68, c, 4000.0 + 10 * i, label="Average balance")
                put(178, c, 300.0 + 2 * i, label="Total income")
                for r, base in ((187, 40.0), (188, 25.0), (189, 12.0),
                                (190, 8.0), (191, 5.0), (192, 3.0)):
                    put(r, c, base + (4.0 if (snow and r == 189) else 0.0),
                        label=f"Expense line {r}")
                put(R[193], c, f=f'=IF({X}27="","",IFERROR(SUM({X}187:{X}192),""))',
                    label="Direct costs")
                for r, base in ((197, 6.0), (198, 4.0), (199, 3.0), (200, 2.0)):
                    put(r, c, base, label=f"Overhead line {r}")
                if snow:
                    put(201, c, 7.5, label="NEW allocated overhead")   # the inserted row
                    put(202, c, 1.5, label="Overhead line 201")
                else:
                    put(201, c, 1.5, label="Overhead line 201")
                last_oh = 202 if snow else 201
                put(R[202], c, f=f'=IF({X}27="","",IFERROR({X}{R[193]}+SUM({X}197:{X}{last_oh}),""))',
                    label="Total costs")
                put(R[204], c, f=f'=IFERROR(IF({X}178-{X}{R[202]}=0,"",{X}178-{X}{R[202]}),"")',
                    label="Net income - month")
                put(R[205], c, f=f'=IF({X}27="","",IFERROR(IF({X}178-{X}{R[202]}=0,"",'
                                 f'SUM($D{R[204]}:{X}{R[204]})),""))', label="Net income - cumulative")
                put(R[207], c, f=f'=IF({X}27="","",IFERROR(SUM({X}{R[205]})/AVERAGE($D68:{X}68),""))',
                    label="Cumulative ROA")
                put(R[209], c, f=f'=IF({X}{R[207]}="","",IFERROR(IF({X}5="M1",{X}{R[207]},'
                                 f'((SUM({X}{R[205]})/AVERAGE($D68:{X}68))/MID({X}$5,2,4))*12),0))',
                    label="Cumulative ROA - Annualized")
        wb.save(path)

    d_old, d_new = demo_dir / "demo_old.xlsx", demo_dir / "demo_snow.xlsx"
    build_demo(d_old, False); build_demo(d_new, True)
    b_old, b_new = ft.load_book(d_old), ft.load_book(d_new)

    # the ROA formula is the same formula, just shifted
    f209 = b_old.formula("Vintage1", 209, 4)
    remapped = ft.remap_rows(f209, row_map)
    print("old D209 :", f209)
    print("remapped :", remapped)
    print("new D210 :", b_new.formula("Vintage1", 210, 4))
    assert re.sub(r"\s+", "", remapped).upper() == \
           re.sub(r"\s+", "", b_new.formula("Vintage1", 210, 4)).upper()
    assert '"M1"' in remapped, 'the string literal "M1" must not be remapped as a reference'

    demo_traces, bad = ft.trace_columns(b_old, b_new, "Vintage1", 209, COLS,
                                        row_map=row_map, label_col=2, max_depth=MAX_DEPTH)
    assert not bad, bad
    assert len(demo_traces) == 24, len(demo_traces)

    print(f"\n{'month':<7}{'old ROA':>11}{'new ROA':>11}{'comps':>7}")
    for tr in demo_traces[:3] + demo_traces[-1:]:
        print(f"{b_old.value('Vintage1', 5, tr.col):<7}{tr.value_old:>11.5%}"
              f"{tr.value_new:>11.5%}{len(tr.components):>7}")

    tr = demo_traces[5]                                     # M6
    base = tr.evaluate_with(set())
    print(f"\nM6 components")
    for c in tr.components:
        print(f"   {c.label:<26}{c.ref_old:>7}{c.old:>9,.2f} -> {c.new:>8,.2f}"
              f"   solo {(tr.evaluate_with({c.key}) - base) * 10_000:+8.2f} bps")

    # M1 takes the short branch; MID reads one digit at M6 and two at M24
    assert abs(demo_traces[0].value_old - 192.5 / 4010.0) < 1e-12, "M1 should pass row 207 through"
    assert all(t.value_old > 0 for t in demo_traces), "every month should compute"
    assert {c.label for c in tr.components} == {"Expense line 189"}, \
        {c.label for c in tr.components}
    assert len(tr.components) == 6, "the cumulative SUM pulls in one cell per month to date"

    widened = [sc for sc in tr.structural if sc.kind == "range widened"]
    assert widened and all(201 in sc.extra_rows for sc in widened), "the inserted row must be reported"
    assert "NEW allocated overhead" in widened[0].extra_detail
    print(f"\nstructural: {widened[0].describe()}")

    for t in demo_traces:
        t.check()                                # the old side must reconcile exactly

    demo_page = rx.write_html(
        rx.build_payload(demo_traces,
                         names=[f"Vintage1 · M{i}" for i in range(1, 25)],
                         groups=["Vintage1"] * 24,
                         items=[f"M{i}" for i in range(1, 25)]),
        demo_dir / "demo_roa_explorer.html",
        title="What moved ROA, component by component",
        subtitle="Generated demo workbooks · row 209 / 210, months M1–M24", sample=True)

    print("\nself-test")
    print("  the real formula chain evaluates, MID reads the month, blanks behave as Excel's,")
    print("  the row map puts 209 against 210, and the widened range is reported not absorbed")
    print(f"  page: {demo_page}")